# الدرس 03 - أنماط التصميم الوكيلية  

في هذا الدرس، نستعرض ثلاثة أنماط تصميم أساسية لبناء وكلاء ذكيين فعالين:  

1. **تعليمات واضحة للوكيل** — صياغة مطالبات دقيقة تحدد الدور وتوجه سلوك الوكيل  
2. **إخراج منظم باستخدام نماذج Pydantic** — ضمان أن يعيد الوكلاء بيانات متوقعة ومحققة  
3. **وكلاء مسؤولية واحدة** — تصميم وكلاء مركزين يقوم كل منهم بعمل واحد بشكل جيد  

سنطبق كل نمط على سيناريو **موصي وجهات السفر**، نبني تدريجياً نظامًا يمكنه اقتراح وجهات، التحقق من التوفر، وإدارة اللوجستيات.  


## الإعداد


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## النمط 1: تعليمات واضحة للوكيل

النمط الأكثر تأثيرًا هو أيضًا الأبسط: كتابة تعليمات واضحة ومفصلة لوكيلك.

التعليمات الجيدة تُحدد:
- **من** هو الوكيل (الشخصية والأسلوب)
- **ماذا** يجب أن يفعل (المسؤوليات خطوة بخطوة)
- **كيف** يجب أن يتصرف (القيود والأسلوب)

أدناه، ننشئ وكيل كونسييرج للسفر مع تعليمات صريحة تشكل كل استجابة ينتجها.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## النمط 2: الإخراج المنظم مع نماذج Pydantic

النص الحر مفيد للمحادثة، لكن الأنظمة اللاحقة تحتاج إلى بيانات منظمة.
من خلال إقران **نماذج Pydantic** مع **دالة أداة**، يمكننا:

- تحديد مخطط دقيق لإخراج الوكيل
- التحقق من صحة الاستجابات تلقائيًا
- دمج نتائج الوكيل في منطق التطبيق بشكل موثوق

المفتاح للتنفيذ هو تمرير `response_format` عند تشغيل الوكيل. هذا يجبر
النموذج على إرجاع كائن `TravelRecommendations` تم التحقق من صحته (متاح في `response.value`)
بدلاً من النص الحر. كما تعيد أداة `get_destination_details` كائنًا محدد النوع
`DestinationRecommendation`، بحيث تبقى البيانات منظمة من البداية للنهاية.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## النمط 3: وكلاء المسؤولية الفردية

تستفيد المهام المعقدة من تقسيم العمل بين عدة وكلاء متخصصين، كل منهم لديه مسؤولية واحدة:

- **خبير الوجهة** الذي يعرف الأماكن وتوافرها
- **مخطط اللوجستيات** الذي يتولى الرحلات الجوية، الفنادق، والجداول الزمنية

هذا يعكس مبدأ هندسة البرمجيات *فصل الاهتمامات* — حيث يكون كل وكيل أسهل في الاختبار، الصيانة، والتحسين بشكل مستقل.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## الملخص

في هذا الدرس طبقنا ثلاثة أنماط تصميم وكيلية على سيناريو موصي سفر:

| النمط | الفكرة الرئيسية | الفائدة |
|---|---|---|
| **تعليمات واضحة** | تحديد الشخصية، والمسؤوليات، والقيود في البداية | سلوك وكيل متناسق ومتوافق مع العلامة التجارية |
| **مخرجات منظمة** | استخدام نماذج Pydantic كصيغة للإجابة | نتائج صحيحة وقابلة للقراءة آليًا |
| **مسؤولية واحدة** | تعيين مهمة واحدة مركزة لكل وكيل | أسهل في الاختبار، والصيانة، والتركيب |

هذه الأنماط تتكامل بشكل طبيعي — يمكنك دمج التعليمات الواضحة مع المخرجات المنظمة داخل وكيل بمسؤولية واحدة لبناء أنظمة متينة وجاهزة للإنتاج.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
